# Pittsburgh CIP PDFs → per-year CSVs

Parses every capital-budget PDF in `PDF/` and writes one CSV per PDF to `CSV/` (same stem: `2008.pdf → CSV/2008.csv`).

**Pipeline:** classify pages (`project` vs `other`) → detect format family per PDF → route each PDF to its format parser → write CSV → print diagnostics.

### Decisions agreed before writing this notebook
| Topic | Rule used here |
|---|---|
| `cip_year` | The fiscal year the plan **starts** (= budget year printed in the PDF). Pittsburgh's fiscal year is the calendar year, and each file is named one year earlier than its budget (`2025.pdf` = *2026 Capital Budget & Six Year Plan* → `cip_year = 2026`). |
| Funding-source rows (CDBG / BOND / PAYGO / CITY / OTHER) | Only the **TOTAL** row feeds the `year_YYYY` columns; the per-source breakdown is not written. |
| `year_YYYY` | One column per year printed in the funding table, **including** the first (current-year) column, e.g. `year_2025` in the 2026 plan. |
| `previous_appropriations` | "Unexpended / Unencumbered Prior Year Funds" (2011–2025) and "Remaining Funds as of …" (2001–2004 scans). |
| `project_total` | **2005–2010:** the summary table's TOTAL column (same value as the single `year_YYYY` column). **All other years: left blank** (lead's decision, Nov 2026 — totals will be calculated later). The printed total ("Total 2026-2031", "Total Cost 2002-2007") is still parsed and checked in section 15; set `WRITE_PRINTED_TOTAL = True` in section 1 to write it again. |
| Units | PDFs print whole dollars → **all money columns are divided by 1,000** (values are in $1,000s; decimals kept). |
| `project_id` | Only a single clean ID printed in the PDF (Project/Grant Number, Project Number, Grant #). **Left blank otherwise** (e.g. 2012–2025 pages, which only show per-source JDE account numbers or nothing). |
| 2005–2010 | No per-project pages exist; each row of the "Capital Project Summary" table is one project with a single `year_<cip_year>` column. |
| 2001–2004 | Scanned pages: page structure and amounts come from the PDF's own OCR text layer. If Tesseract is installed (section 7), names, descriptions and departments are re-read from a page image, and blank Project/Grant Numbers and Total Costs are filled. Department spellings are mapped to the scans' fixed department list. |
| Repeated pages | A project sheet printed twice on consecutive pages (same ID or name, identical amounts) is written once (e.g. 2003 CIP "North Side Underpasses", pp. 153–154). |
| Validator check `project_total ≈ previous_appropriations + Σ year_*` | Not applicable to Pittsburgh: the printed total covers only the planned years (it excludes the current-year column and the unexpended prior funds). Section 15 checks the parsed printed total against the PDF's own rule instead. |
| `start_year`, `end_year` | Not printed in any Pittsburgh format → left blank (not inferred). |

Run the cells top to bottom. Sections 4–6 are inspection cells (page counts, format families, raw-text samples) — read their output before trusting the CSVs.

## 1. Imports and configuration

In [1]:
import re
import csv
import math
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
import pdfplumber          # word-level geometry (x/y/font size) for parsing
import fitz                # PyMuPDF – fast plain-text pass used for page classification

BASE_DIR = Path.cwd()                      # run the notebook from the Pittsburgh/ folder
PDF_DIR = BASE_DIR / "PDF"
CSV_DIR = BASE_DIR / "CSV"
CSV_DIR.mkdir(exist_ok=True)
assert PDF_DIR.is_dir(), f"PDF folder not found: {PDF_DIR} (run the notebook from the Pittsburgh/ folder)"

PDF_FILES = sorted(PDF_DIR.glob("*.pdf"), key=lambda p: p.stem)

UNIT_DIVISOR = 1000        # PDFs print whole dollars; CSVs are written in $1,000s
FOOTER_FRACTION = 0.915    # lines below this fraction of page height are footers (page numbers, banners)

# project_total (lead's decision, Nov 2026): the printed multi-year total is left blank in the CSVs for 2001–2004 and
# 2011–2025 (it will be calculated later from the year columns); the one-year summaries (2005–2010) use their TOTAL.
# The printed total is still parsed and checked in section 15. Set True to write it to the CSVs again.
WRITE_PRINTED_TOTAL = False

SPEC_COLUMNS = [
    "cip_year", "project_type", "source_page", "department", "project_name", "project_id",
    "address_location", "start_year", "end_year", "project_description", "project_justification",
    "previous_appropriations", "project_total",
]

print(f"{len(PDF_FILES)} PDFs found in {PDF_DIR}")
print([p.name for p in PDF_FILES])

25 PDFs found in /Users/markomiocinovic/Downloads/UIL-Research-Work/Pittsburgh/PDF
['2001.pdf', '2002.pdf', '2003.pdf', '2004.pdf', '2005.pdf', '2006.pdf', '2007.pdf', '2008.pdf', '2009.pdf', '2010.pdf', '2011.pdf', '2012.pdf', '2013.pdf', '2014.pdf', '2015.pdf', '2016.pdf', '2017.pdf', '2018.pdf', '2019.pdf', '2020.pdf', '2021.pdf', '2022.pdf', '2023.pdf', '2024.pdf', '2025.pdf']


## 2. Text and geometry helpers

* `page_words` extracts words with position, font size and bold flag. If a page looks like it has dropped spaces
  (many very long tokens), it falls back to rebuilding words from individual characters using horizontal gaps
  relative to font size.
* `group_lines` clusters words into visual text lines (tolerance scales with font size).
* `clean_text` repairs hyphenated line wraps (`word-\nword → wordword`) and collapses whitespace.
* `parse_amount` handles `$1,234`, `-` / `‐` (= 0), `(1,234)` / `-1,234` (negative), `N/A`, `TBD`, `Zero`, blanks.
* `merge_numeric_fragments` re-joins numbers the PDF split into pieces (`1 0,000`, `1 ,300,000`).

In [2]:
_DASH_CHARS = "‐‑‒–—―−﹘﹣－"
_TRANSLATE = {ord(c): "-" for c in _DASH_CHARS}
_TRANSLATE.update({0x00A0: " ", 0xFF1A: ":", 0xFB01: "fi", 0xFB02: "fl", 0x2019: "'", 0x2018: "'",
                   0x201C: '"', 0x201D: '"'})


def norm(s):
    """Normalise dashes, colons, ligatures and quotes."""
    return (s or "").translate(_TRANSLATE)


def clean_text(s):
    """Free-text cleanup: repair hyphenated wraps, collapse whitespace."""
    if not s:
        return ""
    s = norm(s)
    s = re.sub(r"(\w)-\s*\n\s*([a-z])", r"\1\2", s)    # word-\nword -> wordword
    s = re.sub(r"\s*\n\s*", " ", s)
    s = re.sub(r"[ \t]+", " ", s)
    return s.strip()


def _looks_concatenated(words):
    """True if many tokens are suspiciously long runs of letters (spaces dropped by the PDF)."""
    if len(words) < 20:
        return False
    longish = sum(1 for w in words if len(re.sub(r"[^A-Za-z]", "", w["text"])) >= 22)
    return longish / len(words) > 0.04


def _words_from_chars(page):
    """Fallback: rebuild words from characters, splitting where the horizontal gap is large for the font size."""
    chars = sorted(page.chars, key=lambda c: (round(c["top"] / 2), c["x0"]))
    words, cur = [], None
    for ch in chars:
        txt = ch["text"]
        size = ch.get("size", 10) or 10
        if cur is not None:
            same_line = abs(ch["top"] - cur["top"]) <= 0.4 * size
            gap = ch["x0"] - cur["x1"]
            if same_line and gap <= 0.18 * size and not txt.isspace():
                cur["text"] += txt
                cur["x1"] = ch["x1"]
                cur["bottom"] = max(cur["bottom"], ch["bottom"])
                continue
            words.append(cur)
            cur = None
        if txt.isspace():
            continue
        cur = {"text": txt, "x0": ch["x0"], "x1": ch["x1"], "top": ch["top"], "bottom": ch["bottom"],
               "size": size, "fontname": ch.get("fontname", "")}
    if cur is not None:
        words.append(cur)
    return words


def page_words(page):
    """Words with geometry: text, x0, x1, top, bottom, size, bold."""
    raw = page.extract_words(keep_blank_chars=False, use_text_flow=False, extra_attrs=["size", "fontname"])
    if _looks_concatenated(raw):
        raw = _words_from_chars(page)
    out = []
    for w in raw:
        t = norm(w["text"]).strip()
        if not t:
            continue
        out.append({"text": t, "x0": float(w["x0"]), "x1": float(w["x1"]), "top": float(w["top"]),
                    "bottom": float(w["bottom"]), "size": round(float(w.get("size") or 0), 1),
                    "bold": "bold" in (w.get("fontname") or "").lower()})
    return out


def group_lines(words, tol_ratio=0.45):
    """Cluster words into visual lines. Each line: top, bottom, x0, x1, size (max), bold (all), words, text."""
    lines = []
    for w in sorted(words, key=lambda w: (w["top"], w["x0"])):
        placed = False
        for ln in reversed(lines[-4:]):
            tol = max(2.0, tol_ratio * min(w["size"] or 10, ln["size"] or 10))
            if abs(w["top"] - ln["top"]) <= tol:
                ln["words"].append(w)
                ln["size"] = max(ln["size"], w["size"])
                placed = True
                break
        if not placed:
            lines.append({"top": w["top"], "words": [w], "size": w["size"]})
    for ln in lines:
        ln["words"].sort(key=lambda w: w["x0"])
        ln["text"] = " ".join(w["text"] for w in ln["words"])
        ln["x0"] = ln["words"][0]["x0"]
        ln["x1"] = max(w["x1"] for w in ln["words"])
        ln["bottom"] = max(w["bottom"] for w in ln["words"])
        ln["bold"] = all(w["bold"] for w in ln["words"])
    lines.sort(key=lambda l: l["top"])
    return lines


def body_lines(page, words=None, footer_fraction=FOOTER_FRACTION):
    """Lines of a page with footers (page numbers / bottom banners) removed."""
    words = words if words is not None else page_words(page)
    limit = page.height * footer_fraction
    return [ln for ln in group_lines(words) if ln["top"] < limit and not re.fullmatch(r"\d{1,4}", ln["text"])]


def words_text(words):
    return " ".join(w["text"] for w in sorted(words, key=lambda w: (round(w["top"]), w["x0"])))


def lines_text(lines, x_min=None, x_max=None):
    """Join lines (optionally only words inside an x-range) into one cleaned paragraph."""
    parts = []
    for ln in lines:
        ws = [w for w in ln["words"] if (x_min is None or w["x0"] >= x_min) and (x_max is None or w["x0"] < x_max)]
        if ws:
            parts.append(" ".join(w["text"] for w in ws))
    return clean_text("\n".join(parts))


# ---------------------------------------------------------------- numbers
_MISSING_TOKENS = {"N/A", "NA", "TBD", "NONE", "-N/A-"}


def ocr_fix_number(t):
    """Fix common OCR confusions inside tokens that are already mostly numeric (scanned 2001-2004 PDFs)."""
    if re.match(r"^S[\dOol]", t):                       # 'S125,000' -> '$125,000'
        t = "$" + t[1:]
    t = re.sub(r"^-+(?=\$)", "", t)                     # scan speck read as a minus sign: '-$250,000'
    core =re.sub(r"[$,.()\-]", "", t)
    if core and (t.startswith("$") or sum(c.isdigit() for c in core) >= max(1, math.ceil(len(core) / 2))):
        t = t.translate(str.maketrans({"O": "0", "o": "0", "D": "0", "l": "1", "I": "1", "i": "1", "|": "1"}))
    # a period misread for a thousands comma: '$1,700.000' -> '$1,700,000' (real cents have 2 digits)
    t = re.sub(r"(\d)\.(\d{3})(?=$|[,.)])", r"\1,\2", t)
    # a table border read as a trailing '1' after a complete ',ddd' group: '$50,0001' -> '$50,000'
    t = re.sub(r"^(\$?\d{1,3}(?:,\d{3})+)1$", r"\1", t)
    return t


def parse_amount(text, ocr=False):
    """Parse a money cell. Returns float dollars, 0.0 for dashes/'Zero', None for missing/non-numeric."""
    t = norm(text).strip().replace(" ", "")
    if ocr:
        t = ocr_fix_number(t)
    if t in ("", "$"):
        return None
    if re.fullmatch(r"\$?-+\$?", t):                  # '-', '$ -' -> zero
        return 0.0
    up = t.upper().strip("$")
    if up in _MISSING_TOKENS:
        return None
    if up == "ZERO":
        return 0.0
    # cents typed with a comma: '$320,822,13' -> '$320,822.13'
    m = re.fullmatch(r"(\(?-?\$?\d{1,3}(?:,\d{3})*),(\d{2})(\)?)", t)
    if m:
        t = f"{m.group(1)}.{m.group(2)}{m.group(3)}"
    if not re.fullmatch(r"\(?-?\$?\(?-?\d[\d,]*(\.\d+)?\)?", t):
        return None
    neg = (t.startswith("(") and t.endswith(")")) or "-" in t[:3]
    digits = re.sub(r"[^\d.]", "", t)
    try:
        v = float(digits)
    except ValueError:
        return None
    return -v if neg else v


def is_money_token(text, ocr=False):
    """Token that looks like a printed amount ($ sign or thousands separators), not a bare year/day number."""
    t = norm(text).strip()
    if ocr:
        t = ocr_fix_number(t)
    return bool(re.fullmatch(r"\(?-?\$-?\d[\d,]*(\.\d+)?\)?|\(?-?\$?\d{1,3}(,\d{3})+(\.\d+)?\)?", t))


def merge_numeric_fragments(words):
    """Drop stand-alone '$' tokens and re-join digit fragments split by the PDF ('1 0,000' -> '10,000')."""
    ws = [w for w in sorted(words, key=lambda w: w["x0"]) if w["text"] != "$"]
    out = []
    for w in ws:
        if out:
            p = out[-1]
            gap = w["x0"] - p["x1"]
            joined = p["text"] + w["text"]
            if (gap < 0.45 * max(w["size"] or 8, 6)
                    and re.fullmatch(r"\(?\$?[\d,]+", p["text"])
                    and re.fullmatch(r"[\d,]+(\.\d+)?\)?", w["text"])
                    and re.fullmatch(r"\(?\$?(\d{1,3}(,\d{3})+|\d+)(\.\d+)?\)?", joined)):
                out[-1] = dict(p, text=joined, x1=w["x1"])
                continue
        out.append(dict(w))
    return out


def to_thousands(v):
    return None if v is None else round(v / UNIT_DIVISOR, 3)


def fmt_number(v):
    """Format for CSV: no trailing zeros; '' for missing."""
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return ""
    s = f"{v:.3f}".rstrip("0").rstrip(".")
    return "0" if s in ("-0", "") else s


print("helpers ready")

helpers ready


## 3. Shared funding-table reader

Used by formats A, C, D and E. It is **geometric**: it finds the header line with the year columns, takes each column's
x-centre, and assigns every amount on a *labelled data row* to the nearest column. This avoids two failure modes:
sparse rows (empty cells shift amounts left in plain text) and header tokens being read as data.

* Header: a line with ≥ 4 increasing 4-digit years. A year range (`2026-2031`, `2014 - 2019`) on that line or within
  ±14 pt is the **total** column. Years missing from the header (OCR damage) are re-inserted at interpolated positions.
* Data rows: lines below the header up to a stop label (`Unexpended`, `Total Cost`, `Project Description`, …).
  The row label is the text left of the first year column (`CDBG`, `BOND`, `PAYGO`, `CITY`, `OTHER`, `TOTAL`).
* `TOTAL` row → `year_YYYY` and `project_total`. In the scans the total row is unlabelled (last row); if no total row
  exists the source rows are summed and this is recorded in the diagnostics.

In [3]:
YEAR_RE = re.compile(r"^(19[89]\d|20[0-4]\d)$")
RANGE_RE = re.compile(r"^\(?((?:19|20)\d{2})-((?:19|20)\d{2})\)?$")
TABLE_STOP_RE = re.compile(r"unexpended|unencumbered|total\s*cost|remaining|project\s+description|project\s+summ|"
                           r"managing|impact\s+on|location|operating\s+budget", re.I)


def _year_tokens(line, ocr=False):
    """Split a line into year columns and year-range tokens: [('year', y, x0, x1) | ('range', None, x0, x1)]."""
    toks = line["words"]
    out, j = [], 0
    while j < len(toks):
        t = ocr_fix_number(toks[j]["text"]) if ocr else toks[j]["text"]
        if (YEAR_RE.match(t) and j + 2 < len(toks) and toks[j + 1]["text"] == "-"
                and YEAR_RE.match(toks[j + 2]["text"]) and toks[j + 2]["x0"] - toks[j]["x1"] < 30):
            out.append(("range", None, toks[j]["x0"], toks[j + 2]["x1"]))
            j += 3
            continue
        if RANGE_RE.match(t):
            out.append(("range", None, toks[j]["x0"], toks[j]["x1"]))
        elif YEAR_RE.match(t):
            out.append(("year", int(t), toks[j]["x0"], toks[j]["x1"]))
        j += 1
    return out


def find_year_header(lines, ocr=False, min_years=4, start_index=0):
    """Locate the funding-table header. Returns dict(index, top, years=[(year, x_center)], total_center) or None."""
    for i in range(start_index, len(lines)):
        ln = lines[i]
        toks = _year_tokens(ln, ocr)
        years = [(y, (x0 + x1) / 2) for kind, y, x0, x1 in toks if kind == "year"]
        if len(years) < min_years:
            continue
        # Year columns are evenly spaced and consecutive. Index each token by its position on the column grid and
        # let the tokens vote on the starting year, so a misprinted ("2016, 2016") or OCR-damaged ("2008" for 2006)
        # header year, or a dropped one, does not break the table.
        cs = [c for _, c in years]
        gaps = sorted(b - a for a, b in zip(cs, cs[1:]))
        step = gaps[len(gaps) // 2]
        if step < 20:
            continue
        idx = [round((c - cs[0]) / step) for c in cs]
        if len(set(idx)) != len(idx) or max(idx) + 1 > 10:
            continue
        votes = Counter(y - k for (y, _), k in zip(years, idx))
        start, n_votes = votes.most_common(1)[0]
        if n_votes < max(2, math.ceil(len(years) / 2)):
            continue
        actual = dict(zip(idx, cs))
        filled = [(start + k, actual.get(k, cs[0] + step * k)) for k in range(max(idx) + 1)]
        # total column: a year range on this line or a nearby line, right of the last year
        total_center = None
        for other in lines:
            if abs(other["top"] - ln["top"]) <= 14:
                for kind, _, x0, x1 in _year_tokens(other, ocr):
                    c = (x0 + x1) / 2
                    if kind == "range" and c > filled[-1][1] + 15:
                        total_center = c
        if total_center is None:
            for other in lines:
                if abs(other["top"] - ln["top"]) <= 14:
                    for w in other["words"]:
                        c = (w["x0"] + w["x1"]) / 2
                        if w["text"].upper() == "TOTAL" and c > filled[-1][1] + 15:
                            total_center = c
        return {"index": i, "top": ln["top"], "years": filled, "total_center": total_center}
    return None


def read_funding_table(lines, hdr, ocr=False, stop_re=TABLE_STOP_RE, max_depth=230):
    """Read labelled rows under a header. Returns list of {label, values{year|'total': dollars}, top, bottom}."""
    cols = [(y, c) for y, c in hdr["years"]]
    if hdr["total_center"] is not None:
        cols.append(("total", hdr["total_center"]))
    first_center = min(c for _, c in cols)
    rows = []
    for ln in lines[hdr["index"] + 1:]:
        if ln["top"] - hdr["top"] > max_depth:
            break
        if stop_re.search(ln["text"]):
            break
        toks = _year_tokens(ln, ocr)
        if toks and all(w["text"] in ("-", "Total", "TOTAL") or YEAR_RE.match(w["text"]) or RANGE_RE.match(w["text"])
                        for w in ln["words"]):
            continue                                       # second header line ("2026-2031", "Total")
        label_words = [w for w in ln["words"] if w["x1"] < first_center - 12 and re.search(r"[A-Za-z]", w["text"])]
        label = " ".join(w["text"] for w in label_words).upper()
        label_edge = max((w["x1"] for w in label_words), default=-1)
        values = {}
        for w in merge_numeric_fragments([w for w in ln["words"] if w["x0"] > label_edge]):
            v = parse_amount(w["text"], ocr)
            if v is None:
                continue
            c = (w["x0"] + w["x1"]) / 2
            key, dist = min(((k, abs(c - cc)) for k, cc in cols), key=lambda t: t[1])
            if key not in values or dist < values[key][1]:
                values[key] = (v, dist)
        values = {k: v for k, (v, _) in values.items()}
        if not values and not label:
            continue
        if not values and rows and not re.fullmatch(r"[A-Z ]{3,12}", label):
            break                                          # prose after the table
        rows.append({"label": label, "values": values, "top": ln["top"], "bottom": ln["bottom"]})
    return rows


def pick_total_row(rows):
    """Return (values, how). Prefer a 'TOTAL' row; else last unlabelled row (scans); else sum of source rows."""
    for r in rows:
        if re.fullmatch(r"TOTALS?", re.sub(r"[^A-Z]", "", r["label"])) and r["values"]:
            return r["values"], "TOTAL row"
    unlabelled = [r for r in rows if not r["label"] and r["values"]]
    if unlabelled:
        return unlabelled[-1]["values"], "unlabelled total row"
    labelled = [r for r in rows if r["values"]]
    if len(labelled) == 1:
        return labelled[0]["values"], "single source row"
    if labelled:
        summed = defaultdict(float)
        for r in labelled:
            for k, v in r["values"].items():
                summed[k] += v
        return dict(summed), "sum of source rows"
    return {}, "no table values"


def table_zone(hdr, rows):
    """Vertical extent of the funding table (used to keep table text out of free-text fields)."""
    bottom = max([r["bottom"] for r in rows], default=hdr["top"] + 10)
    return (hdr["top"] - 22, bottom + 4)


def in_zone(line, zone):
    return zone is not None and zone[0] <= line["top"] <= zone[1]


def new_record(cip_year, page_no):
    rec = {c: None for c in SPEC_COLUMNS}
    rec.update({"cip_year": cip_year, "source_page": page_no, "_years": {}, "_diag": {}})
    return rec


def apply_table(rec, hdr, rows):
    """Copy TOTAL-row values into the record (dollars; converted to thousands at write time)."""
    vals, how = pick_total_row(rows)
    for y, _ in hdr["years"]:
        rec["_years"][y] = vals.get(y)
    if "total" in vals:
        rec["project_total"] = vals["total"]
    rec["_diag"]["total_row"] = how


print("funding-table reader ready")

funding-table reader ready


## 4. Page classification (`project` vs `other`) and `cip_year`

Every page gets a fast plain-text pass (PyMuPDF). A page is a **project page** only if it carries the per-project
signals of its format; summary/index/glossary/operating-budget pages fail these tests:

| Signature | Required signals on the page |
|---|---|
| **A** (scanned 2001–2004 sheets) | `SOURCE` table label + `Managing Department` + (`Total Cost` or `Proposed / Six-Year`) |
| **B** (2005–2010 summary table) | table header with `DESCRIPTION` + `GRANT` + `CDBG` + `Total` on the page |
| **C** (2011 sheets) | `Project Name:` + `Project Number:` + `SOURCE` funding table |
| **D** (2012–2013 sheets) | a bare `Project:` label + `Priority Level` + `SOURCE` funding table (rules out "Department Summary" pages) |
| **E** (2014–2025 sheets) | exactly one `Functional Area:` + `Responsible Department` + `CDBG`/`TOTAL` funding table |
| **E-cont** | `Deliverables and Objectives` page that directly follows an E project page (stitched to that project) |

Format-level checks (e.g. a real year header in the funding table) are applied again inside each parser, so a summary
page that slips through still produces no row.

In [4]:
SIGNATURES = {
    "A": lambda t: (re.search(r"S\s*O\s*U\s*R\s*C\s*E", t, re.I) is not None
                    and re.search(r"manag\w*\s+dep", t, re.I) is not None
                    and re.search(r"total\s*cost|six.?year|proposed", t, re.I) is not None),
    "B": lambda t: (re.search(r"\bDESCRIPTION\b", t) is not None and re.search(r"\bGRANT\b", t) is not None
                    and re.search(r"\bCDBG\b", t) is not None and re.search(r"\bTOTAL\b|\bTotal\b", t) is not None),
    "C": lambda t: (re.search(r"Project\s+Name\s*:", t) is not None and re.search(r"Project\s+Number\s*:", t) is not None
                    and re.search(r"\bSOURCE\b", t) is not None),
    "D": lambda t: (re.search(r"(?m)^\s*Project\s*:\s*$", t) is not None and re.search(r"Priority\s+Level", t) is not None
                    and re.search(r"\bSOURCE\b", t) is not None),
    "E": lambda t: (len(re.findall(r"Functional\s+Area\s*:", t)) == 1
                    and re.search(r"Responsible\s+Department", t) is not None
                    and re.search(r"\bCDBG\b", t) is not None and re.search(r"\bTOTAL\b", t) is not None),
}
E_CONT_RE = re.compile(r"Deliverables\s+and\s+Objectives", re.I)


def page_texts(pdf_path):
    with fitz.open(pdf_path) as doc:
        return [norm(pg.get_text()) for pg in doc]


def detect_cip_year(texts, stem):
    """Budget (= starting fiscal) year printed in the document; falls back to file-name year + 1."""
    hits = Counter()
    for t in texts:
        for m in re.finditer(r"\b((?:19|20)\d{2})\s+(?:Revised\s+)?(?:Operating\s*(?:and|&)\s*)?Capital\s+(?:Budget|Project)",
                             t, re.I):
            hits[int(m.group(1))] += 1
        for m in re.finditer(r"Plan\s*\(\s*((?:19|20)\d{2})\s*-\s*(?:19|20)\d{2}\s*\)", t, re.I):
            hits[int(m.group(1))] += 1
    fallback = int(stem) + 1 if stem.isdigit() else None
    if hits:
        year, _ = hits.most_common(1)[0]
        return year, "detected"
    return fallback, "fallback (file-name year + 1)"


def detect_cip_year_cover_first(texts, stem):
    """Cover pages first (body text also cites the previous year's budget), then the whole document."""
    for scope, ts in (("cover", texts[:3]), ("document", texts)):
        year, how = detect_cip_year(ts, stem)
        if how == "detected":
            return year, f"detected ({scope})"
    return detect_cip_year([], stem)


def classify_pdf(pdf_path):
    texts = page_texts(pdf_path)
    sig_hits = {k: [i for i, t in enumerate(texts) if f(t)] for k, f in SIGNATURES.items()}
    # the format family is the signature that matches the most pages
    fam = max(sig_hits, key=lambda k: (len(sig_hits[k]), -"ABCDE".index(k)))
    if not sig_hits[fam]:
        fam = None
    labels = ["other"] * len(texts)
    for i in sig_hits.get(fam, []):
        labels[i] = "project"
    if fam == "E":   # stitch 'Deliverables and Objectives' continuation pages onto the preceding project page
        for i in range(1, len(texts)):
            if labels[i] == "other" and E_CONT_RE.search(texts[i]) and labels[i - 1] in ("project", "project-cont"):
                labels[i] = "project-cont"
    cip_year, how = detect_cip_year_cover_first(texts, pdf_path.stem)
    return {"path": pdf_path, "texts": texts, "labels": labels, "family_by_signature": fam,
            "sig_counts": {k: len(v) for k, v in sig_hits.items()}, "cip_year": cip_year, "cip_year_source": how}


CLASSIFIED = {}
for pdf_path in PDF_FILES:
    CLASSIFIED[pdf_path.stem] = classify_pdf(pdf_path)

count_rows = []
for stem, c in CLASSIFIED.items():
    n_main = c["labels"].count("project")
    n_cont = c["labels"].count("project-cont")
    count_rows.append({"pdf": f"{stem}.pdf", "pages": len(c["labels"]), "project_pages": n_main + n_cont,
                       "project_start_pages": n_main, "continuation_pages": n_cont,
                       "cip_year": c["cip_year"], "cip_year_source": c["cip_year_source"],
                       **{f"sig_{k}": v for k, v in c["sig_counts"].items()}})
PAGE_COUNTS = pd.DataFrame(count_rows)
print("Project page count per PDF (project_pages = start pages + stitched continuation pages):")
display(PAGE_COUNTS)

Project page count per PDF (project_pages = start pages + stitched continuation pages):


,pdf,pages,project_pages,project_start_pages,continuation_pages,cip_year,cip_year_source,sig_A,sig_B,sig_C,sig_D,sig_E
0,2001.pdf,223,198,198,0,2002,detected (document),198,0,0,0,0
1,2002.pdf,285,235,235,0,2003,detected (cover),235,0,0,0,0
2,2003.pdf,85,52,52,0,2004,detected (document),52,0,0,0,0
3,2004.pdf,80,72,72,0,2005,detected (cover),72,0,0,0,0
4,2005.pdf,20,15,15,0,2006,detected (cover),0,15,0,0,0
5,2006.pdf,230,5,5,0,2007,detected (document),0,5,0,0,0
6,2007.pdf,257,7,7,0,2008,detected (cover),0,7,0,0,0
7,2008.pdf,333,7,7,0,2009,detected (cover),0,7,0,0,0
8,2009.pdf,285,7,7,0,2010,detected (document),0,7,0,0,0
9,2010.pdf,374,6,6,0,2011,detected (document),0,6,0,0,0


## 5. Format families

The first confirmed project page of every PDF is sampled and fingerprinted (orientation, field labels present, number of
year columns in the funding table, whether the total column is a year range). PDFs with the same fingerprint family
are grouped, and every PDF is routed to that family's parser.

| Family | Years | Layout |
|---|---|---|
| **A** | 2001–2004 | Scanned landscape sheet, one project per page. Division banner top-left, category line, centered project title top-right, `SOURCE` table with 7 year columns (current + 6 planned) and an *unlabelled* total row, `Total Cost YYYY-YYYY`, `Remaining Funds as of …`, `Managing Department - …`, `Project/Grant Number`. |
| **B** | 2005–2010 | Not per-project sheets: a "Capital Project Summary" table, one row per project (`DEPT`/`PROJECT`/`DESCRIPTION`/`GRANT #`/`CDBG`/`BOND`/`CITY`/`SUBTOTAL`/`OTHER`/`TOTAL`) for the budget year only. Rows are bottom-aligned (name + description wrap above the amounts row). |
| **C** | 2011 | Landscape sheet: banner `City of Pittsburgh … <Department>`, `Project Name:` / `Project Number:` on one line, `Project Description and Justification:`, `SOURCE` table (7 years + `YYYY - YYYY` total), `Unexpended Prior Year Funds (as of …)`, `Location / Census Tract:`. |
| **D** | 2012–2013 | Landscape sheet: same banner, labels `Project:` / `Description:` / `Location:` / `Priority Level:` with values in a second column, same `SOURCE` table and `Unexpended / Unencumbered …` line. |
| **E** | 2014–2025 | Portrait sheet: large title, `Functional Area:` / `Project Type:` / `Responsible Department:` / `Project Manager:`, "Capital Improvement Program" table (`Source`, 7 years, `Total YYYY-YYYY`), `Project Description`, `Project Justification`, `Operating Budget Impact`, `Unexpended/Unencumbered Prior Year Funds`. The following "Deliverables and Objectives" page (Location / District columns) is stitched on for `address_location`. |

In [5]:
def fingerprint(pdf_path, page_index):
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_index]
        words = page_words(page)
        lines = group_lines(words)
        text = " ".join(ln["text"] for ln in lines)
        hdr = find_year_header(lines, ocr=True)
        return {
            "orientation": "landscape" if page.width > page.height else "portrait",
            "has_text_layer": len(words) > 0,
            "year_columns": len(hdr["years"]) if hdr else 0,
            "total_is_range": bool(hdr and hdr["total_center"] is not None),
            "labels": ", ".join(lbl for lbl, rx in [
                ("Functional Area", r"Functional Area"), ("Project Name:", r"Project Name\s*:"),
                ("Project Number:", r"Project Number\s*:"), ("Project:", r"(^|\s)Project\s*:"),
                ("Priority Level", r"Priority Level"), ("Managing Department", r"Manag\w*\s+Dep"),
                ("GRANT #", r"GRANT\s*#"), ("Total Cost", r"Total\s+Cost")] if re.search(rx, text, re.I)),
        }


FAMILY_DESCRIPTION = {
    "A": "scanned per-project sheet (2001-2004 style)",
    "B": "capital project summary table (2005-2010 style)",
    "C": "Project Name / Project Number sheet (2011 style)",
    "D": "Project / Description / Location / Priority Level sheet (2012-2013 style)",
    "E": "Functional Area / Project Type sheet + Deliverables page (2014-2025 style)",
}

def run_format_detection():
    rows = []
    for stem, c in CLASSIFIED.items():
        first = next((i for i, l in enumerate(c["labels"]) if l == "project"), None)
        fp = fingerprint(c["path"], first) if first is not None else {}
        rows.append({"pdf": f"{stem}.pdf", "format": c["family_by_signature"] or "none",
                     "first_project_page": None if first is None else first + 1, **fp})
    df = pd.DataFrame(rows)
    print("PDFs per format family:")
    for fam, grp in df.groupby("format"):
        print(f"  Format {fam} — {FAMILY_DESCRIPTION.get(fam, 'no project pages found')}: {', '.join(grp['pdf'])}")
    return df


FORMAT_TABLE = run_format_detection()
display(FORMAT_TABLE)

PDFs per format family:
  Format A — scanned per-project sheet (2001-2004 style): 2001.pdf, 2002.pdf, 2003.pdf, 2004.pdf
  Format B — capital project summary table (2005-2010 style): 2005.pdf, 2006.pdf, 2007.pdf, 2008.pdf, 2009.pdf, 2010.pdf
  Format C — Project Name / Project Number sheet (2011 style): 2011.pdf
  Format D — Project / Description / Location / Priority Level sheet (2012-2013 style): 2012.pdf, 2013.pdf
  Format E — Functional Area / Project Type sheet + Deliverables page (2014-2025 style): 2014.pdf, 2015.pdf, 2016.pdf, 2017.pdf, 2018.pdf, 2019.pdf, 2020.pdf, 2021.pdf, 2022.pdf, 2023.pdf, 2024.pdf, 2025.pdf


,pdf,format,first_project_page,orientation,has_text_layer,year_columns,total_is_range,labels
0,2001.pdf,A,11,landscape,True,7,False,"Managing Department, Total Cost"
1,2002.pdf,A,9,landscape,True,7,False,"Managing Department, Total Cost"
2,2003.pdf,A,6,landscape,True,7,False,"Managing Department, Total Cost"
3,2004.pdf,A,4,landscape,True,7,False,"Managing Department, Total Cost"
4,2005.pdf,B,3,landscape,True,0,False,GRANT #
5,2006.pdf,B,215,landscape,True,0,False,GRANT #
6,2007.pdf,B,251,landscape,True,0,False,GRANT #
7,2008.pdf,B,321,landscape,True,0,False,GRANT #
8,2009.pdf,B,276,landscape,True,0,False,GRANT #
9,2010.pdf,B,350,landscape,True,0,False,GRANT #


## 6. Raw-text sanity check (validate before regex-ing)

Prints the visual lines of one sample project page per format so you can confirm spacing, colons and table alignment.
Each line shows its top y-coordinate and font size.

In [6]:
def show_raw_page(stem, page_index, max_lines=45):
    c = CLASSIFIED[stem]
    with pdfplumber.open(c["path"]) as pdf:
        page = pdf.pages[page_index]
        lines = group_lines(page_words(page))
    print(f"===== {stem}.pdf  page {page_index + 1}  (format {c['family_by_signature']}) =====")
    for ln in lines[:max_lines]:
        print(f"  y={ln['top']:6.1f} size={ln['size']:4.1f} | {ln['text'][:150]}")
    print()


_shown = set()
for stem, c in CLASSIFIED.items():
    fam = c["family_by_signature"]
    if fam and fam not in _shown:
        first = c["labels"].index("project")
        show_raw_page(stem, first)
        _shown.add(fam)

===== 2001.pdf  page 11  (format A) =====
  y=  36.2 size=19.0 | Community Development
  y=  52.7 size=13.0 | Greater Pittsburgh Community Food Bank
  y= 127.3 size=17.0 | I I
  y= 134.4 size= 8.0 | PROPOSED Proposed Six-Year Capital Improvement Plan (2002-2007)
  y= 144.5 size= 8.0 | BUDGET
  y= 164.9 size= 8.0 | 2001 2002 2003 2004 2005 2006 2007
  y= 169.2 size= 8.0 | SOURCE
  y= 195.1 size= 8.0 | CDBG $0 $30,000 $30,000 $30,000 $30,000 $30,000 $30,000
  y= 223.0 size= 8.0 | $0 $30,000 $30,000 $30,000 $30,000 $30,000 $30,000
  y= 265.0 size= 8.0 | Total Cost 2002-2007 $180,000
  y= 512.3 size=13.0 | Managing Department - City Council ProjecUGrant Number
  y= 527.1 size=11.0 | New
  y= 555.7 size= 6.0 | caU56
  y= 568.4 size=10.0 | 4

===== 2005.pdf  page 3  (format B) =====
  y=  76.4 size=44.0 | Pittsburgh City Council
  y= 267.8 size= 8.8 | PROJECT
  y= 279.0 size= 8.8 | DEPARTMENT PROJECT DESCRIPTION GRANT # CDBG CITY OTHER Total
  y= 289.9 size= 9.6 | City Council
  y= 310.5 siz

## 7. Optional Tesseract re-OCR for the scanned sheets (2001–2004)

The 2001–2004 PDFs are scans whose built-in text layer was produced by a weak OCR pass ("CDSG", "vveed&seed",
"Neig-hborhood"). When Tesseract is installed, the parser keeps using the text layer for **page structure and the
funding table** (positions are reliable and the amounts were validated), but re-reads the text of a few **fields**
from a 300-dpi render of the page:

* `project_name`, `project_description`, `department`: re-OCR text is used when it is at least as clean as the
  text layer (share of word-like tokens). Department names are then mapped to the scans' fixed department list.
* `project_id`, `project_total`: re-OCR only **fills blanks** (e.g. 2003 "Project/Grant Number" boxes and "Total Cost"
  amounts the text layer garbled); values the text layer already read are never replaced.

**Setup (once):** `conda install -c conda-forge tesseract` and `pip install pytesseract` (Homebrew users:
`brew install tesseract`). The notebook also finds Tesseract in the usual conda/Homebrew folders when Jupyter (e.g. in
VS Code) starts without them on PATH, and points it at its language files. If Tesseract is missing or
`USE_TESSERACT = False`, the notebook uses the text layer only and says why. Re-OCR adds roughly 1–2 s per scanned
page (≈ 10–15 min for all four scanned PDFs).

In [7]:
import difflib

USE_TESSERACT = True          # set to False to use only the PDF's own OCR text layer
TESSERACT_DPI = 300

import os
import shutil

# Jupyter (e.g. in VS Code) may start without the conda/Homebrew folders on PATH, so look in the usual places too.
_TESSERACT_CANDIDATES = [
    shutil.which("tesseract"),
    "/opt/anaconda3/bin/tesseract", "/opt/miniconda3/bin/tesseract",
    os.path.expanduser("~/anaconda3/bin/tesseract"), os.path.expanduser("~/miniconda3/bin/tesseract"),
    "/opt/homebrew/bin/tesseract", "/usr/local/bin/tesseract",
]

try:
    import pytesseract
    from PIL import Image
    _tess_cmd = next((p for p in _TESSERACT_CANDIDATES if p and os.path.exists(p)), None)
    if _tess_cmd:
        pytesseract.pytesseract.tesseract_cmd = _tess_cmd
        # a conda/Homebrew tesseract started from outside its environment may not find its language files
        _tessdata = os.path.join(os.path.dirname(os.path.dirname(os.path.realpath(_tess_cmd))), "share", "tessdata")
        if not os.environ.get("TESSDATA_PREFIX") and os.path.exists(os.path.join(_tessdata, "eng.traineddata")):
            os.environ["TESSDATA_PREFIX"] = _tessdata + os.sep
    _TESS_VERSION = pytesseract.get_tesseract_version()
    # smoke test: one real OCR call, so a broken install is reported here instead of failing on every page
    pytesseract.image_to_string(Image.new("L", (60, 30), 255))
    TESSERACT_AVAILABLE = True
except Exception as _exc:     # module missing or tesseract binary not on PATH
    TESSERACT_AVAILABLE = False
    _TESS_VERSION = repr(_exc)

REOCR_ACTIVE = USE_TESSERACT and TESSERACT_AVAILABLE
REOCR_STATS = Counter()
if REOCR_ACTIVE:
    print(f"Tesseract {_TESS_VERSION} found at {pytesseract.pytesseract.tesseract_cmd} -> re-OCR of scanned-sheet fields is ON")
else:
    print("Tesseract re-OCR is OFF -> using the PDFs' own OCR text layer only.",
          "" if not USE_TESSERACT else f"\n  reason: {_TESS_VERSION}"
          "\n  enable with: conda install -c conda-forge tesseract  and  pip install pytesseract")


class PageImage:
    """Whole page rendered once in its *displayed* orientation (some scans are stored rotated 90/270 degrees);
    crops are given in pdfplumber page coordinates and scaled to image pixels."""

    def __init__(self, fitz_page, width, height, dpi=TESSERACT_DPI):
        pix = fitz_page.get_pixmap(dpi=dpi, colorspace=fitz.csGRAY)
        self.img = Image.frombytes("L", (pix.width, pix.height), pix.samples)
        self.sx = pix.width / width
        self.sy = pix.height / height

    def ocr(self, bbox, psm=6):
        x0, top, x1, bottom = bbox
        box = (max(0, int(x0 * self.sx)), max(0, int(top * self.sy)),
               min(self.img.width, int(x1 * self.sx)), min(self.img.height, int(bottom * self.sy)))
        if box[2] - box[0] < 8 or box[3] - box[1] < 8:
            return ""
        try:
            return norm(pytesseract.image_to_string(self.img.crop(box), config=f"--psm {psm}")).strip()
        except Exception as exc:
            REOCR_STATS["tesseract errors"] += 1
            if REOCR_STATS["tesseract errors"] == 1:
                print("First Tesseract error:", str(exc)[:300])
            return ""


def bbox_of(words, pad=4):
    return (min(w["x0"] for w in words) - pad, min(w["top"] for w in words) - pad,
            max(w["x1"] for w in words) + pad, max(w["bottom"] for w in words) + pad)


_WORDLIKE = re.compile(r"^[(\"']?([A-Za-z][a-z]+|[A-Z]{2,}|[A-Z][a-z]*[A-Z][a-z]+|\d[\d,.$%]*|[A-Za-z]\.?|&|-|/)"
                       r"(?:'s)?[)\"'.,;:]*$")


def text_quality(text):
    """Share of tokens that look like real words/numbers (OCR debris scores low)."""
    toks = (text or "").split()
    return sum(bool(_WORDLIKE.match(t)) for t in toks) / len(toks) if toks else 0.0


def _letters(s):
    return re.sub(r"[^a-z]", "", (s or "").lower())


def better_text(original, candidate, field, min_similarity=0.0):
    """Use the re-OCR text when it is clean (>= 75% word-like tokens) and at least as clean as the text layer, or clearly
    cleaner than the text layer. With min_similarity, the re-OCR
    text must also resemble the text-layer version (guards against a clipped crop read as e.g. 'eee eee')."""
    if not candidate:
        return original
    if min_similarity and len(_letters(original)) >= 6:
        lo, lc = _letters(original), _letters(candidate)
        sim = difflib.SequenceMatcher(None, lo, lc).ratio()
        # words may be in a different order, or the text layer may have extra words: also accept when nearly all
        # letters of a reasonably long re-OCR name occur in the text-layer version
        contained = sum((Counter(lc) & Counter(lo)).values()) / max(len(lc), 1)
        if sim < min_similarity and not (contained >= 0.8 and len(lc) >= 8):
            REOCR_STATS[f"{field}: re-OCR rejected (unlike text layer)"] += 1
            return original
    q_new, q_old = text_quality(candidate), text_quality(original)
    if not original or (q_new >= q_old and q_new >= 0.75) or q_new > q_old + 0.15:
        REOCR_STATS[f"{field}: re-OCR text used"] += 1
        return candidate
    REOCR_STATS[f"{field}: text layer kept"] += 1
    return original


# Managing departments that appear on the 2001-2004 sheets; OCR spelling variants are mapped onto these.
SCAN_DEPARTMENTS = [
    "City Controller", "City Council", "City Planning", "Engineering & Construction", "Equipment Leasing Authority",
    "Finance", "General Services", "Housing Authority", "Human Relations Commission", "Information Systems",
    "Mayor's Office", "Parking Authority", "Parks & Recreation", "Personnel & Civil Service", "Public Safety",
    "Public Works", "Urban Redevelopment Authority", "Water & Sewer Authority",
]
_DEPT_KEYS = {}


def _dept_key(s):
    return re.sub(r"[^a-z]", "", norm(s).lower().replace("&", "and"))


def canonical_department(text):
    """'Mayor•s Office' / "Mayor'~ Office" -> "Mayor's Office"; 'CityPlannlnq' -> 'City Planning'."""
    if not text:
        return text
    if not _DEPT_KEYS:
        _DEPT_KEYS.update({_dept_key(d): d for d in SCAN_DEPARTMENTS})
    key = _dept_key(text)
    if key in _DEPT_KEYS:
        return _DEPT_KEYS[key]
    match = difflib.get_close_matches(key, list(_DEPT_KEYS), n=1, cutoff=0.8)
    return _DEPT_KEYS[match[0]] if match else text

Tesseract 5.2.0 found at /opt/anaconda3/bin/tesseract -> re-OCR of scanned-sheet fields is ON


## 8. Parser — Format A (scanned sheets, 2001–2004)

Page structure comes from the scan's OCR text layer, so every step is position-based and tolerant of noise:
* **project_name**: large text in the top-right title area (above the category line and the small print); OCR debris
  tokens (`I`, `~`, `--u--J`) are removed.
* **project_type**: the category line under the top-left division banner (e.g. *General Government*,
  *Mayor's Local Option*), read from the left half of the page only. The banner itself (e.g. *Community Development*)
  is a division heading and is not used.
* **project_description**: the "Project Summary" block (2003–2004) or, if that block is missing or empty, the
  small-print text under the title (2001–2002).
* **department**: `Managing Department - …`, mapped to the fixed department list.  **project_id**: value under
  `Project/Grant Number` (blank if `New`).
* **project_total**: `Total Cost YYYY-YYYY $…`.  **previous_appropriations**: `Remaining Funds as of … $…`.
* Fields are re-read with Tesseract when it is available (section 7).

In [8]:
_KEEP_PUNCT = {"-", "&", "/"}


def _clean_ocr_title(text):
    """Remove OCR debris tokens ('I', '~', '--u--J', '[') but keep real words, numbers and '-', '&', '/' separators."""
    keep = []
    text = re.sub(r"#\s+(\d)", r"#\1", norm(text))              # 'Viaduct # 1' -> 'Viaduct #1'
    for tok in text.split():
        if re.fullmatch(r"#\d+[A-Za-z]?", tok) or (tok.isdigit() and keep and
                                                     re.fullmatch(r"(?i)phase|no\.?|district|ward|unit|stage|area", keep[-1])):
            keep.append(tok)
            continue
        if tok.count("_") >= 3:
            # letter-by-letter OCR ('N_e_i_g-,-h_b_o_r_h_o_o_d_P_a_r_k...'): drop the glue, re-split at case changes
            tok = re.sub(r"[_~,\-]+", "", tok)
            tok = re.sub(r"(?<=[a-z]{2})(?=[A-Z])", " ", tok)
            keep.extend(tok.split())
            continue
        if tok in _KEEP_PUNCT:
            keep.append(tok)
            continue
        letters = sum(ch.isalnum() for ch in tok)
        if letters == 0 or letters / len(tok) < 0.6:
            continue
        if len(tok) == 1 and tok not in ("A",):
            continue
        tok = re.sub(r"-{2,}$", "", re.sub(r"^-+", "", tok.strip("~|[]{}·•_")))   # keep 'Communications-'
        keep.append(tok)
    keep = [k for k in keep if k]
    # titles start capitalised and do not end in a 1-2 letter lowercase fragment ('yo - Census Products', '... ee')
    while keep and (keep[0] in _KEEP_PUNCT or re.fullmatch(r"[a-z]{1,2}", keep[0])):
        keep.pop(0)
    while keep and (keep[-1] in _KEEP_PUNCT or re.fullmatch(r"[a-z]{1,2}", keep[-1])):
        keep.pop()
    return clean_text(" ".join(keep)).strip(" .,")


def _clean_ocr_prose(text):
    """Lighter filter for descriptions: drop tokens that are mostly OCR line-art / punctuation."""
    keep = [tok for tok in (text or "").split()
            if (sum(ch.isalnum() for ch in tok) / len(tok) >= 0.5 or tok in _KEEP_PUNCT | {"$"})
            and len(re.findall(r"[~_]", tok)) < 3]          # letter-by-letter debris ('~O~p~e~ra~ti~ng') is unreadable
    out = clean_text(" ".join(keep)).strip(" -~|[]")
    out = re.sub(r"^(?:[\dIl|]\s+)+", "", out)               # box borders read as leading '1' / 'I'
    return out if re.search(r"[A-Za-z]{3}", out) else ""


_SCAN_ID_RE = re.compile(r"\b([A-Z]?\d{6,8})\b")       # e.g. 2216302, G221502 (not page codes like COU2)


def parse_format_A(pdf_path, page_indices, cip_year):
    records = []
    fdoc = fitz.open(pdf_path) if REOCR_ACTIVE else None
    with pdfplumber.open(pdf_path) as pdf:
        for pi in page_indices:
            page = pdf.pages[pi]
            words = page_words(page)
            lines = group_lines(words)                     # keep footers: project id sits near the bottom
            pimg = None
            if fdoc is not None:
                try:
                    pimg = PageImage(fdoc[pi], page.width, page.height)
                except Exception:
                    REOCR_STATS["page render errors"] += 1
            hdr = find_year_header(lines, ocr=True)
            rec = new_record(cip_year, pi + 1)
            if hdr is not None:
                rows = read_funding_table(lines, hdr, ocr=True)
                apply_table(rec, hdr, rows)
                table_top = hdr["top"]
            else:
                # funding table missing from the OCR layer: keep the project, leave its year columns blank
                rec["_diag"]["total_row"] = "funding table unreadable (OCR)"
                cost = next((ln for ln in lines if re.search(r"total\s*cost", ln["text"], re.I)), None)
                table_top = cost["top"] - 60 if cost else page.height * 0.3

            banner_line = next((ln for ln in lines if re.search(r"proposed|six.?year|capital\s*imp", ln["text"], re.I)
                                and ln["top"] < table_top), None)
            title_limit = banner_line["top"] - 2 if banner_line else table_top - 40
            mid = page.width * 0.42

            # left half, above the table: [division banner, category line, ...] (lines of pure debris are skipped)
            left = []
            for ln in lines:
                if ln["top"] >= title_limit:
                    continue
                txt = _clean_ocr_title(" ".join(w["text"] for w in ln["words"] if w["x0"] < mid and w["size"] >= 11))
                if txt:
                    left.append((ln, txt))
            category = left[1] if len(left) >= 2 else None
            if category:
                rec["project_type"] = category[1]

            # the title sits above the category line and above the small-print text in the right half
            # small print is 7-9 pt; title words can be OCR'd a little smaller than the rest of the title
            small_right = [ln for ln in lines if ln["top"] < title_limit and ln["x0"] >= mid and ln["size"] < 9.5
                           and len(re.findall(r"[A-Za-z]", ln["text"])) >= 5]   # letters may be OCR'd 'O~p~e~r...'

            limits = [title_limit]
            if category:
                limits.append(category[0]["top"] + 14)
            if small_right:
                limits.append(small_right[0]["top"] - 1)
            name_limit = min(limits)
            title_words = [w for ln in lines if ln["top"] < name_limit
                           for w in ln["words"] if w["x0"] >= mid and w["size"] >= 9.5]
            # OCR boxes of one title line can sit at slightly different heights: group loosely, read left to right
            title_text = " ".join(ln["text"] for ln in group_lines(title_words, tol_ratio=0.8))
            rec["project_name"] = _clean_ocr_title(title_text) or None
            if pimg is not None and title_words:
                n_title_lines = len(group_lines(title_words, tol_ratio=0.8))
                lettered = [w for w in title_words if len(re.findall(r"[A-Za-z]", w["text"])) >= 2]
                cands = []
                # two crops: all title words, and only lettered words (debris marks like '[~~-' can stretch the
                # first one over the box border); keep the cleaner reading, then the one closer to the text layer
                for crop_words in ([title_words, lettered] if lettered and len(lettered) < len(title_words)
                                   else [title_words]):
                    tb = bbox_of(crop_words, 5)
                    # wider sideways (text-layer boxes clip last letters), never below the small print under the title
                    tb = (tb[0] - 15, tb[1], min(tb[2] + 25, page.width - 5), min(tb[3], name_limit))
                    got = [l for l in pimg.ocr(tb).splitlines() if l.strip()][:n_title_lines]
                    txt = _clean_ocr_title(" ".join(got))
                    if txt:
                        sim = difflib.SequenceMatcher(None, _letters(rec["project_name"]), _letters(txt)).ratio()
                        cands.append((text_quality(txt), sim, txt))
                best = max(cands)[2] if cands else None
                rec["project_name"] = better_text(rec["project_name"], best, "name", min_similarity=0.6)

            desc_lines = []
            summ = next((k for k, ln in enumerate(lines) if re.search(r"project\s+summ", ln["text"], re.I)), None)
            if summ is not None:
                for ln in lines[summ + 1:]:
                    if re.search(r"manag\w*\s+dep", ln["text"], re.I):
                        break
                    desc_lines.append(ln)
                rec["project_description"] = _clean_ocr_prose(lines_text(desc_lines, x_max=page.width * 0.72)) or None
            if not rec["project_description"]:             # no / empty "Project Summary" box: small print under title
                small_words = [w for ln in lines if ln["top"] < title_limit for w in ln["words"]
                               if w["x0"] >= mid and w["size"] < 9.5]
                desc_lines = group_lines(small_words)
                rec["project_description"] = _clean_ocr_prose(lines_text(desc_lines)) or None
            desc_words = [w for ln in desc_lines for w in ln["words"] if w["x0"] < page.width * 0.92]
            if pimg is not None and desc_words:
                if summ is not None and desc_lines and desc_lines[0]["top"] > lines[summ]["top"]:
                    dbox = bbox_of(desc_words, 5)                  # "Project Summary" block (2003-2004)
                else:
                    # small print under the title (2001-2002): crop the band between the title and the
                    # "Proposed Six-Year" banner, so debris marks inside the title box cannot pull the title in
                    first_title = group_lines(title_words, tol_ratio=0.8)[0] if title_words else None
                    d_top = (first_title["bottom"] + 3) if first_title else min(w["top"] for w in desc_words) - 5
                    lettered = [w["top"] for w in desc_words if len(re.findall(r"[A-Za-z]", w["text"])) >= 3]
                    if lettered:                               # start at the real small print, below the title glyphs
                        d_top = max(d_top, min(lettered) - 4)
                    dbox = (mid - 10, d_top, page.width - 8, max(d_top + 12, title_limit - 2))
                rec["project_description"] = better_text(rec["project_description"],
                                                         _clean_ocr_prose(pimg.ocr(dbox)) or None, "description")

            for ln in lines:
                txt = ln["text"]
                if re.search(r"total\s*cost", txt, re.I):
                    amts = [w for w in merge_numeric_fragments(ln["words"]) if is_money_token(w["text"], ocr=True)]
                    rem_x = next((w["x0"] for w in ln["words"] if re.match(r"remain", w["text"], re.I)), None)
                    before = [w for w in amts if rem_x is None or w["x0"] < rem_x]
                    if before:
                        rec["project_total"] = parse_amount(before[0]["text"], ocr=True)
                    elif pimg is not None:
                        # amount garbled in the text layer: re-read the box right of "Total Cost YYYY-YYYY"
                        lab = [w for w in ln["words"] if re.search(r"cost|\d{4}", w["text"], re.I) and w["x0"] < mid]
                        if lab:
                            x0 = max(w["x1"] for w in lab) + 2
                            x1 = (rem_x - 4) if rem_x else x0 + 170
                            got = pimg.ocr((x0, ln["top"] - 6, x1, ln["bottom"] + 6), psm=7)
                            m = re.search(r"\$\s?[\d,.]{3,}", got.replace(" ", ""))
                            if m and is_money_token(m.group(0), ocr=True):
                                rec["project_total"] = parse_amount(m.group(0), ocr=True)
                                REOCR_STATS["project_total: filled by re-OCR"] += 1
                if re.search(r"remain\w*\s+funds", txt, re.I):
                    rem_x = next(w["x0"] for w in ln["words"] if re.match(r"remain", w["text"], re.I))
                    amts = [w for w in merge_numeric_fragments(ln["words"])
                            if w["x0"] > rem_x and is_money_token(w["text"], ocr=True)]
                    if amts:
                        rec["previous_appropriations"] = parse_amount(amts[-1]["text"], ocr=True)
                m = re.search(r"manag\w*\s+dep\w*\s*[-:]*\s*(.*)", txt, re.I)
                if m and rec["department"] is None:
                    dept_words = [w for w in ln["words"] if w["x0"] < page.width * 0.72]
                    strip_label = lambda s: re.sub(r"(?i)^.*?manag\w*\s+dep\w*\s*[-:]*\s*", "", s)
                    dtxt = _clean_ocr_title(strip_label(" ".join(w["text"] for w in dept_words))) or None
                    if pimg is not None and dept_words:
                        dtxt = better_text(dtxt, _clean_ocr_title(strip_label(pimg.ocr(bbox_of(dept_words, 4), psm=7)))
                                           or None, "department")
                    rec["department"] = canonical_department(dtxt)
                if re.search(r"grant", txt, re.I) and ln["x1"] > page.width * 0.7 and rec["project_id"] is None:
                    label_words = [w for w in ln["words"] if re.search(r"grant|proj|number", w["text"], re.I)
                                   and w["x0"] > page.width * 0.6]
                    gx = min(w["x0"] for w in label_words) - 40 if label_words else page.width * 0.7
                    for ln2 in lines:
                        if ln["top"] < ln2["top"] <= ln["top"] + 35:
                            for w in ln2["words"]:
                                t = ocr_fix_number(w["text"]) if re.search(r"\d", w["text"]) else w["text"]
                                if w["x0"] >= gx and re.fullmatch(r"[A-Z]?\d{5,8}", t):
                                    rec["project_id"] = t
                                    break
                        if rec["project_id"]:
                            break

            # Project/Grant Number box sits at the right end of the "Managing Department" line; its label is often
            # garbled in the text layer, so re-read the whole box area when the ID is still blank
            mline = next((ln for ln in lines if re.search(r"manag\w*\s+dep", ln["text"], re.I)), None)
            if rec["project_id"] is None and pimg is not None and mline is not None:
                got = pimg.ocr((page.width * 0.72, mline["top"] - 12, page.width - 8, mline["top"] + 45), psm=6)
                got = re.sub(r"(?<=\d)[Oo]|[Oo](?=\d)", "0", got)
                mid_ = _SCAN_ID_RE.search(re.sub(r"[^A-Za-z0-9]+", " ", got))
                if mid_:
                    rec["project_id"] = mid_.group(1)
                    REOCR_STATS["project_id: filled by re-OCR"] += 1

            if rec["project_name"] and (hdr is not None or rec["project_total"] is not None or rec["department"]):
                records.append(rec)
    if fdoc is not None:
        fdoc.close()
    return records


print("Format A parser ready")

Format A parser ready


## 9. Parser — Format B (capital project summary tables, 2005–2010)

Column boundaries come from the table header words on each page (`DEPT`, `PROJECT`, `DESCRIPTION`, `GRANT #`,
money columns). Rows are **bottom-aligned**: the amounts sit on the last line of each row, with the project name and
description wrapping above. So a row is every line after the previous amounts line, up to and including the next one.
* A printed `GRANT #` (or `NEW` / `N/A` in that column) also closes a row: projects whose funding was cut in a revision have a grant number but no
  amounts; they are kept with an empty year value instead of being merged into the next row.
* Bold lines without amounts (`City Council`, `CIS:`) are department headings; a blank `DEPT` cell continues the
  department of the row above.
* `… Total`, `Subtotal`, `TOTAL FUNDS ALLOCATED`, `Grand Total` lines are skipped (they are not projects).
* The row's `TOTAL` amount → `year_<cip_year>`. `GRANT #` → `project_id` (`N/A` / `Multiple` → blank).

In [9]:
B_SUBTOTAL_RE = re.compile(r"\btotal\b|\bsubtotal\b|grand|funds\s+allocated", re.I)
GRANT_ID_RE = re.compile(r"[A-Z]?\d{6,8}")


def _left_edges(lines, x_limit):
    """Frequent left edges of text segments (columns are left-aligned; headings may be centred above them)."""
    edges = Counter()
    for ln in lines:
        prev_x1 = None
        for w in ln["words"]:
            if w["x0"] >= x_limit:
                break
            if prev_x1 is None or w["x0"] - prev_x1 > 8:
                edges[round(w["x0"] / 3) * 3] += 1
            prev_x1 = w["x1"]
    return sorted(x for x, n in edges.items() if n >= 3)


def _snap(header_x0, edges, floor):
    """Largest frequent left edge at or slightly right of the heading, but right of the previous column."""
    cands = [e for e in edges if floor + 10 < e <= header_x0 + 6]
    return max(cands) if cands else header_x0


def _b_columns(header_line, page_lines):
    ws = header_line["words"]
    up = [(w["text"].upper().strip(":#"), w) for w in ws]
    get = lambda name: next((w for t, w in up if t == name), None)
    proj, desc, grant = get("PROJECT"), get("DESCRIPTION"), get("GRANT")
    dept = next((w for t, w in up if t in ("DEPT", "DEPARTMENT", "DEPARTMENTS", "ALL")), None)
    if desc is None or grant is None:
        return None
    money = {t: (w["x0"] + w["x1"]) / 2 for t, w in up if t in ("CDBG", "BOND", "CITY", "SUBTOTAL", "OTHER", "TOTAL")}
    if "TOTAL" not in money:
        return None
    value_x0 = min(money.values()) - 30
    edges = _left_edges([ln for ln in page_lines if ln["top"] > header_line["top"]], value_x0)
    first_x = dept["x0"] if dept else ws[0]["x0"]
    if proj is not None and desc["x0"] - proj["x1"] < 25:          # "PROJECT DESCRIPTION" is one column heading
        name_x0 = _snap(first_x, edges, -100)
        desc_x0 = _snap(proj["x0"], edges, name_x0)
        dept_rng = None
    else:
        dept_x0 = _snap(dept["x0"], edges, -100) if dept is not None else None
        name_x0 = _snap(proj["x0"] if proj else first_x, edges, dept_x0 if dept_x0 is not None else -100)
        desc_x0 = _snap(desc["x0"], edges, name_x0)
        dept_rng = (dept_x0 - 4, name_x0 - 3) if dept_x0 is not None and dept_x0 < name_x0 - 10 else None
    near = [e for e in edges if grant["x0"] - 25 <= e <= grant["x0"] + 6 and e > desc_x0 + 10]
    grant_x0 = min(near + [grant["x0"]]) - 3
    prev_money = max((c for t, c in money.items() if t != "TOTAL"), default=money["TOTAL"] - 60)
    return {
        "dept": dept_rng,
        "name": (name_x0 - 3, desc_x0 - 3),
        "desc": (desc_x0 - 3, grant_x0),
        "grant": (grant_x0, min(money.values()) - 18),
        "money": money,
        "total_split": (prev_money + money["TOTAL"]) / 2,
        "value_x0": value_x0,
    }


def _in(w, rng):
    return rng is not None and rng[0] <= w["x0"] < rng[1]


def parse_format_B(pdf_path, page_indices, cip_year):
    records = []
    department = None
    with pdfplumber.open(pdf_path) as pdf:
        for pi in page_indices:
            page = pdf.pages[pi]
            lines = body_lines(page, footer_fraction=0.975)     # table rows can run close to the page bottom
            cols, group = None, []
            for ln in lines:
                up = ln["text"].upper()
                if "DESCRIPTION" in up and "CDBG" in up and "TOTAL" in up:
                    cols, group = _b_columns(ln, lines), []
                    continue
                if cols is None:
                    continue
                amounts = [w for w in merge_numeric_fragments([w for w in ln["words"] if w["x0"] >= cols["value_x0"]])
                           if parse_amount(w["text"]) is not None]
                total_tok = [w for w in amounts if (w["x0"] + w["x1"]) / 2 > cols["total_split"]]
                left_text = " ".join(w["text"] for w in ln["words"] if w["x0"] < cols["grant"][0])
                # a printed grant number also closes a row: projects whose funding was cut in a revision
                # (e.g. "$20,000 bond funding eliminated") have a grant # but no amounts
                # (only when the project name also ends on this line: some projects list two grant numbers)
                has_grant_id = (any(_in(w, cols["grant"]) and (GRANT_ID_RE.fullmatch(w["text"])
                                                               or w["text"].upper() in ("NEW", "N/A"))
                                    for w in ln["words"])
                                and any(_in(w, cols["name"]) for w in ln["words"]))
                if not total_tok and not has_grant_id:
                    if ln["bold"] and all(w["x0"] < cols["desc"][0] for w in ln["words"]) and not B_SUBTOTAL_RE.search(left_text):
                        department, group = clean_text(ln["text"]).rstrip(":").strip(), []
                    else:
                        group.append(ln)
                    continue
                if B_SUBTOTAL_RE.search(left_text):
                    group = []
                    continue
                row_lines = group + [ln]
                group = []
                name = lines_text(row_lines, x_min=cols["name"][0], x_max=cols["name"][1])
                if not name:
                    continue
                rec = new_record(cip_year, pi + 1)
                rec["project_name"] = name
                rec["project_description"] = lines_text(row_lines, x_min=cols["desc"][0], x_max=cols["desc"][1]) or None
                dept_here = lines_text([ln], x_min=cols["dept"][0], x_max=cols["dept"][1]) if cols["dept"] else ""
                if dept_here:
                    department = dept_here                  # a blank DEPT cell continues the department above
                rec["department"] = department
                gids = [w["text"] for r_ln in row_lines for w in r_ln["words"]
                        if _in(w, cols["grant"]) and GRANT_ID_RE.fullmatch(w["text"])]
                rec["project_id"] = gids[0] if gids else None       # 'N/A', 'NEW', 'Multiple' -> blank
                if total_tok:
                    rec["_years"][cip_year] = parse_amount(total_tok[-1]["text"])
                    rec["_diag"]["total_row"] = "summary-table TOTAL column"
                else:
                    rec["_years"][cip_year] = None
                    rec["_diag"]["total_row"] = "summary-table row without amounts"
                records.append(rec)
    return records


print("Format B parser ready")

Format B parser ready


## 10. Shared labelled-field extractor (formats C and D)

Labels sit in a left column and values in a right column; a value may start on the label's line or one line above
(vertically centred) and continues until the next label. The funding-table zone is excluded so table text never leaks
into a field. The department comes from the right-hand side of the `City of Pittsburgh … <Department>` banner.

In [10]:
def label_fields(lines, label_res, value_x, exclude_zone=None, tol=4.0):
    anchors, seen = [], set()
    for ln in lines:
        left = " ".join(w["text"] for w in ln["words"] if w["x1"] <= value_x + 2)
        for key, rx in label_res.items():
            if key not in seen and re.match(rx, left, re.I):
                anchors.append((ln["top"], key))
                seen.add(key)
                break
    anchors.sort()
    out = {}
    for k, (top, key) in enumerate(anchors):
        end = anchors[k + 1][0] if k + 1 < len(anchors) else float("inf")
        if exclude_zone and top < exclude_zone[0] < end:
            end = min(end, exclude_zone[0] + tol)
        sel = [ln for ln in lines if top - tol <= ln["top"] < end - tol and not in_zone(ln, exclude_zone)]
        out[key] = lines_text(sel, x_min=value_x - 2) or None
    return out


def banner_department(lines, page_width):
    for ln in lines[:6]:
        if re.search(r"City\s+of\s+Pittsburgh", ln["text"], re.I):
            right = [w for w in ln["words"] if w["x0"] > page_width * 0.5]
            if right:
                return clean_text(" ".join(w["text"] for w in right))
    return None


def unexpended_amount(lines, zone_top=None):
    for ln in lines:
        if re.search(r"unexpended", ln["text"], re.I) and (zone_top is None or ln["top"] > zone_top):
            amts = [w for w in merge_numeric_fragments(ln["words"]) if parse_amount(w["text"]) is not None
                    and not YEAR_RE.match(w["text"]) and not re.fullmatch(r"\d{1,2},?", w["text"])]
            if amts:
                return parse_amount(amts[-1]["text"])
    return None


def _value_x(lines, label_rx, default):
    """x where values start = first word right of the label on the label's line."""
    for ln in lines:
        m = re.match(label_rx, ln["text"], re.I)
        if m:
            after = [w for w in ln["words"] if w["x0"] > ln["words"][0]["x0"] + 1 and not w["text"].endswith(":")
                     and not re.match(label_rx, w["text"], re.I)]
            label_end = max(w["x1"] for w in ln["words"] if w["text"].endswith(":") or w is ln["words"][0])
            after = [w for w in after if w["x0"] > label_end + 3]
            if after:
                return min(after[0]["x0"], default)
    return default

## 11. Parser — Format C (2011)

In [11]:
C_LABELS = {
    "name": r"Project\s+Name",
    "description": r"Project\s+Description",
    "impact": r"Impact\s+on\s+Operating",
    "location": r"Location",
    "other_sources": r"Specify",
    "useful_life": r"Estimated\s+Useful",
}


def parse_format_C(pdf_path, page_indices, cip_year):
    records = []
    with pdfplumber.open(pdf_path) as pdf:
        for pi in page_indices:
            page = pdf.pages[pi]
            lines = body_lines(page)
            hdr = find_year_header(lines)
            if hdr is None:
                continue
            rows = read_funding_table(lines, hdr)
            zone = table_zone(hdr, rows)
            zone = (zone[0], max(zone[1], next((ln["bottom"] + 2 for ln in lines
                                                 if re.search(r"unexpended", ln["text"], re.I)), zone[1])))
            rec = new_record(cip_year, pi + 1)
            apply_table(rec, hdr, rows)
            # 'Project Number: <id>' shares the line with the project name: read it, then drop those words
            for ln in lines:
                m = re.search(r"Project\s+Number\s*:?\s*(\S+)?", ln["text"], re.I)
                if m:
                    num_x = next(w["x0"] for w in ln["words"] if w["text"].lower() == "project"
                                 and w["x0"] > page.width * 0.5) if any(w["text"].lower() == "project" and w["x0"] > page.width * 0.5
                                                                        for w in ln["words"]) else None
                    if num_x is not None:
                        pid = (m.group(1) or "").strip()
                        rec["project_id"] = pid if GRANT_ID_RE.fullmatch(pid) else None
                        ln["words"] = [w for w in ln["words"] if w["x0"] < num_x - 2]
                        ln["text"] = " ".join(w["text"] for w in ln["words"])
                    break
            value_x = _value_x(lines, r"Project\s+Name\s*:", page.width * 0.25)
            f = label_fields(lines, C_LABELS, value_x, exclude_zone=zone)
            rec["project_name"] = f.get("name")
            rec["project_description"] = f.get("description")      # PDF combines description and justification
            rec["address_location"] = f.get("location")
            rec["department"] = banner_department(lines, page.width)
            rec["previous_appropriations"] = unexpended_amount(lines)
            if rec["project_name"]:
                records.append(rec)
    return records


print("Format C parser ready")

Format C parser ready


## 12. Parser — Format D (2012–2013)

In [12]:
D_LABELS = {
    "name": r"Project\s*:",
    "description": r"Description\s*:",
    "location": r"Location\s*:",
    "priority": r"Priority\s+Level",
}


def parse_format_D(pdf_path, page_indices, cip_year):
    records = []
    with pdfplumber.open(pdf_path) as pdf:
        for pi in page_indices:
            page = pdf.pages[pi]
            lines = body_lines(page)
            hdr = find_year_header(lines)
            if hdr is None:
                continue
            rows = read_funding_table(lines, hdr)
            rec = new_record(cip_year, pi + 1)
            apply_table(rec, hdr, rows)
            value_x = _value_x(lines, r"Project\s*:", page.width * 0.2)
            f = label_fields(lines, D_LABELS, value_x, exclude_zone=table_zone(hdr, rows))
            rec["project_name"] = f.get("name")
            rec["project_description"] = f.get("description")
            rec["address_location"] = f.get("location")
            rec["department"] = banner_department(lines, page.width)
            rec["previous_appropriations"] = unexpended_amount(lines)
            if rec["project_name"]:
                records.append(rec)
    return records


print("Format D parser ready")

Format D parser ready


## 13. Parser — Format E (2014–2025)

* **project_name**: the largest-font text above `Functional Area:` (the centred title, the project anchor).
* Inline fields `Functional Area:`, `Project Type:`, `Responsible Department:` support both `Label: value` and
  `Label\nvalue`. `project_type` = `Project Type` (e.g. *Continuing, Capital Project*).
* Section headings `Project Description`, `Project Justification`, `Operating Budget Impact`,
  `Unexpended/Unencumbered Prior Year Funds` delimit free-text blocks (table zone and footer excluded).
* Continuation pages ("Deliverables and Objectives") are stitched onto the project: their `Location` (+ `District`)
  column values become `address_location`, de-duplicated and joined with `; `.

In [13]:
E_SECTIONS = {
    "description": r"Project\s+Description",
    "justification": r"Project\s+Justification",
    "impact": r"Operating\s+Budget\s+Impact",
    "prior": r"Unexpended\s*/?\s*Unencumbered\s+Prior\s+Year\s+Funds",
    "jde": r"JDE\s+Numbers?",
    "cip": r"Capital\s+Improvement\s+Program",
    "deliverables": r"(?:\d{4}\s+)?Deliverables\s+and\s+Objectives",
}
E_INLINE = {
    "functional_area": r"Functional\s+Area",
    "project_type": r"Project\s+Type",
    "department": r"Responsible\s+Department",
    "manager": r"Project\s+Manager",
}


def _is_e_label(text):
    return (any(re.match(rf"\s*{rx}", text, re.I) for rx in E_INLINE.values())
            or any(re.fullmatch(rf"\s*{rx}\s*:?\s*", text, re.I) for rx in E_SECTIONS.values()))


def _inline_value(lines, k, label_rx):
    ln = lines[k]
    m = re.match(rf"\s*{label_rx}\s*[:\-]?\s*(.*)$", ln["text"], re.I)
    if not m:
        return None
    val = m.group(1).strip()
    if not val:                                              # two-line layout: Label\nvalue
        if k + 1 < len(lines) and not _is_e_label(lines[k + 1]["text"]):
            val = lines[k + 1]["text"].strip()
        return clean_text(val) or None
    # same-line value that wraps onto following lines aligned with the value column
    n_label_words = len(ln["text"][:m.start(1)].split())
    value_x = ln["words"][n_label_words]["x0"] if n_label_words < len(ln["words"]) else None
    j = k + 1
    while value_x is not None and j < len(lines):
        nxt = lines[j]
        if (_is_e_label(nxt["text"]) or abs(nxt["x0"] - value_x) > 6
                or nxt["top"] - lines[j - 1]["top"] > 1.6 * max(nxt["size"], 8)):
            break
        val += " " + nxt["text"]
        j += 1
    return clean_text(val) or None


def _deliverable_locations(page):
    lines = body_lines(page)
    hdr_k = next((k for k, ln in enumerate(lines) if re.search(r"\bLocation\b", ln["text"])
                  and re.search(r"Deliverable|Cost|Fund", ln["text"], re.I)), None)
    if hdr_k is None:
        return []
    hdr = lines[hdr_k]
    heads = [(w["text"], w["x0"]) for w in hdr["words"]]
    loc_x = next(x for t, x in heads if t.lower() == "location")
    later = [x for t, x in heads if x > loc_x + 5]
    loc_rng = (loc_x - 3, (later[0] - 3) if later else page.width)
    dist_x = next((x for t, x in heads if t.lower() == "district"), None)
    dist_rng = None
    if dist_x is not None:
        after = [x for t, x in heads if x > dist_x + 5]
        dist_rng = (dist_x - 3, (after[0] - 3) if after else page.width)
    body = []
    for ln in lines[hdr_k + 1:]:
        if re.search(r"Deliverables\s+are\s+tentative", ln["text"], re.I) or (ln["size"] >= 13 and ln["bold"]):
            break
        body.append(ln)
    # rows are anchored on lines that carry a cost / fund value at the right; other lines attach to the nearest anchor
    cost_x = max(x for _, x in heads)
    anchors = [ln for ln in body if any(w["x0"] >= cost_x - 15 for w in ln["words"])] or body
    groups = defaultdict(list)
    for ln in body:
        a = min(anchors, key=lambda a: abs(a["top"] - ln["top"]))
        groups[id(a)].append(ln)
    out = []
    for a in anchors:
        g = sorted(groups[id(a)], key=lambda l: l["top"])
        loc = lines_text(g, x_min=loc_rng[0], x_max=loc_rng[1])
        dist = lines_text(g, x_min=dist_rng[0], x_max=dist_rng[1]) if dist_rng else ""
        # keep only district values (the Fund column can sit close to the District column)
        dist = ", ".join(dict.fromkeys(re.findall(r"District\s*\d+|City\s*-?\s*Wide|Multiple|Various", dist, re.I)))
        if loc and dist and dist.lower() not in loc.lower():
            loc = f"{loc} ({dist})"
        if loc:
            out.append(loc)
    return out


def parse_format_E(pdf_path, page_indices, cip_year, cont_map):
    records = []
    with pdfplumber.open(pdf_path) as pdf:
        for pi in page_indices:
            page = pdf.pages[pi]
            lines = body_lines(page)
            fa_k = next((k for k, ln in enumerate(lines) if re.match(r"\s*Functional\s+Area", ln["text"], re.I)), None)
            hdr = find_year_header(lines)
            if fa_k is None or hdr is None:
                continue
            rows = read_funding_table(lines, hdr)
            zone = table_zone(hdr, rows)
            rec = new_record(cip_year, pi + 1)
            apply_table(rec, hdr, rows)

            # title = largest text above 'Functional Area:' (skips smaller department/area banners)
            cands = [ln for ln in lines[:fa_k] if not re.search(r"Capital\s+Improvement\s+Program|City\s+of\s+Pittsburgh",
                                                                ln["text"], re.I) and not in_zone(ln, zone)]
            if cands:
                big = max(ln["size"] for ln in cands)
                rec["project_name"] = clean_text(" ".join(ln["text"] for ln in cands if ln["size"] >= big - 0.6))

            inline = {}
            for k, ln in enumerate(lines):
                for key, rx in E_INLINE.items():
                    if key not in inline and re.match(rf"\s*{rx}", ln["text"], re.I):
                        inline[key] = _inline_value(lines, k, rx)
            rec["project_type"] = inline.get("project_type")
            rec["department"] = inline.get("department")

            heads = sorted([(ln["top"], key) for ln in lines for key, rx in E_SECTIONS.items()
                            if re.fullmatch(rf"\s*{rx}\s*:?\s*", ln["text"], re.I)])
            sections = {}
            for k, (top, key) in enumerate(heads):
                end = heads[k + 1][0] if k + 1 < len(heads) else float("inf")
                sel = [ln for ln in lines if top < ln["top"] < end and not in_zone(ln, zone)]
                sections[key] = sel
            rec["project_description"] = lines_text(sections.get("description", [])) or None
            rec["project_justification"] = lines_text(sections.get("justification", [])) or None
            prior_txt = lines_text(sections.get("prior", []))
            m = re.search(r"\(?-?\$?\s*[\d,]+(?:\.\d+)?\)?", prior_txt)
            rec["previous_appropriations"] = parse_amount(m.group(0)) if m else None

            locs = []
            for ci in cont_map.get(pi, []):
                locs += _deliverable_locations(pdf.pages[ci])
            seen, uniq = set(), []
            for l in locs:
                if l.lower() not in seen:
                    seen.add(l.lower())
                    uniq.append(l)
            rec["address_location"] = "; ".join(uniq) or None
            rec["_diag"]["continuation_pages"] = [c + 1 for c in cont_map.get(pi, [])]
            records.append(rec)
    return records


print("Format E parser ready")

Format E parser ready


## 14. Route every PDF through its parser and write `CSV/<stem>.csv`

* Columns: the 13 spec columns, then `year_YYYY` for every year printed in that PDF's funding tables (sorted).
* Money columns are converted to $1,000s. Missing values are written as empty cells.
* UTF-8, RFC-4180 (comma-separated, `\r\n` line endings, fields quoted when they contain commas, quotes or newlines).
* A project sheet printed twice on consecutive pages (same ID or name, identical amounts) is written once; dropped
  pages are listed in the diagnostics. Projects that merely share a grant number are kept.

In [14]:
PARSERS = {"A": parse_format_A, "B": parse_format_B, "C": parse_format_C, "D": parse_format_D, "E": parse_format_E}


def parse_pdf(stem):
    c = CLASSIFIED[stem]
    fam = c["family_by_signature"]
    if fam is None:
        return fam, []
    labels = c["labels"]
    starts = [i for i, l in enumerate(labels) if l == "project"]
    if fam == "E":
        cont_map, cur = defaultdict(list), None
        for i, l in enumerate(labels):
            if l == "project":
                cur = i
            elif l == "project-cont" and cur is not None:
                cont_map[cur].append(i)
            else:
                cur = None if l == "other" else cur
        return fam, PARSERS[fam](c["path"], starts, c["cip_year"], cont_map)
    return fam, PARSERS[fam](c["path"], starts, c["cip_year"])


DROPPED_DUPLICATES = []


def _name_key(s):
    return re.sub(r"[^a-z]", "", (s or "").lower())


def drop_repeated_pages(stem, records):
    """Drop a project sheet that repeats the sheet on the previous page (same project printed twice).
    Required: consecutive pages, identical amounts (all years, total, prior funds), and the same project ID or a
    near-identical name. Projects that only share a grant number or name keep both rows."""
    kept = []
    for r in records:
        prev = kept[-1] if kept else None
        if prev is not None and r["source_page"] == prev["source_page"] + 1:
            same_money = (r["_years"] == prev["_years"] and r["project_total"] == prev["project_total"]
                          and r["previous_appropriations"] == prev["previous_appropriations"])
            sim = difflib.SequenceMatcher(None, _name_key(r["project_name"]), _name_key(prev["project_name"])).ratio()
            same_project = (r["project_id"] and r["project_id"] == prev["project_id"] and sim >= 0.6) or sim >= 0.9
            if same_money and same_project:
                if text_quality(r["project_name"]) > text_quality(prev["project_name"]):
                    prev["project_name"] = r["project_name"]           # keep the cleaner OCR of the two copies
                    prev["project_description"] = r["project_description"] or prev["project_description"]
                DROPPED_DUPLICATES.append({"pdf": f"{stem}.pdf", "dropped_page": r["source_page"],
                                           "kept_page": prev["source_page"], "project_name": prev["project_name"]})
                continue
        kept.append(r)
    return kept


def csv_project_total(r, fam):
    """Value written to project_total: the TOTAL column for one-year summaries (format B), else blank unless
    WRITE_PRINTED_TOTAL is on."""
    if fam == "B":
        vals = [v for v in r["_years"].values() if v is not None]
        return vals[0] if len(vals) == 1 else None
    return r.get("project_total") if WRITE_PRINTED_TOTAL else None


def records_to_frame(records, fam):
    years = sorted({y for r in records for y in r["_years"]})
    out = []
    for r in records:
        row = {}
        for col in SPEC_COLUMNS:
            v = csv_project_total(r, fam) if col == "project_total" else r.get(col)
            if col in ("previous_appropriations", "project_total"):
                v = fmt_number(to_thousands(v))
            row[col] = "" if v is None else v
        for y in years:
            row[f"year_{y}"] = fmt_number(to_thousands(r["_years"].get(y)))
        out.append(row)
    cols = SPEC_COLUMNS + [f"year_{y}" for y in years]
    return pd.DataFrame(out, columns=cols)


RESULTS = {}
for stem in CLASSIFIED:
    fam, recs = parse_pdf(stem)
    if fam != "B":                                         # one project per page -> a repeated page is a duplicate
        recs = drop_repeated_pages(stem, recs)
    df = records_to_frame(recs, fam)
    out_path = CSV_DIR / f"{stem}.csv"
    df.to_csv(out_path, index=False, encoding="utf-8", quoting=csv.QUOTE_MINIMAL, lineterminator="\r\n")
    RESULTS[stem] = {"format": fam, "records": recs, "df": df}
    print(f"{stem}.pdf  format {fam or '-'}  -> {out_path.name}: {len(df)} rows, "
          f"year columns {[c[5:] for c in df.columns if c.startswith('year_')]}")

2001.pdf  format A  -> 2001.csv: 198 rows, year columns ['2001', '2002', '2003', '2004', '2005', '2006', '2007']
2002.pdf  format A  -> 2002.csv: 234 rows, year columns ['2002', '2003', '2004', '2005', '2006', '2007', '2008']
2003.pdf  format A  -> 2003.csv: 52 rows, year columns ['2003', '2004', '2005', '2006', '2007', '2008', '2009']
2004.pdf  format A  -> 2004.csv: 72 rows, year columns ['2004', '2005', '2006', '2007', '2008', '2009', '2010']
2005.pdf  format B  -> 2005.csv: 103 rows, year columns ['2006']
2006.pdf  format B  -> 2006.csv: 48 rows, year columns ['2007']
2007.pdf  format B  -> 2007.csv: 75 rows, year columns ['2008']
2008.pdf  format B  -> 2008.csv: 84 rows, year columns ['2009']
2009.pdf  format B  -> 2009.csv: 90 rows, year columns ['2010']
2010.pdf  format B  -> 2010.csv: 120 rows, year columns ['2011']
2011.pdf  format C  -> 2011.csv: 114 rows, year columns ['2011', '2012', '2013', '2014', '2015', '2016', '2017']
2012.pdf  format D  -> 2012.csv: 125 rows, year col

## 15. Diagnostics

* **Per-PDF summary**: rows written vs project start pages, and how many projects' planned years add up to the printed
  multi-year total (a mismatch points to a column-mapping or OCR problem, not necessarily a missing value).
* **Format B check**: for the 2005–2010 summary tables there is no per-project total, so the sum of all parsed rows
  is compared with the document's printed `Grand Total` / `TOTAL ALL PROJECTS` line instead.
* **Re-OCR summary** (scans only) and **dropped repeated pages**.
* **Null rates per column** (% empty) — use these to tell legitimately missing fields (e.g. `start_year` is never
  printed; `project_id` is blank after 2011 by design) from parser bugs (e.g. a sudden 100% empty `project_name`).

In [15]:
summary = []
for stem, res in RESULTS.items():
    recs = res["records"]
    n_ok = n_bad = n_blank = 0
    for r in recs:
        if r.get("project_total") is None:
            continue
        ys = sorted(r["_years"])
        plan = [r["_years"][y] for y in ys[1:]]                 # first column = current year
        if not plan or any(v is None for v in plan):
            n_blank += 1                                        # out-years not printed (2003-2004 scans)
            continue
        if abs(sum(plan) - r["project_total"]) <= max(2.0, 0.001 * abs(r["project_total"])):
            n_ok += 1
        else:
            n_bad += 1
    how = Counter(r["_diag"].get("total_row") for r in recs)
    summary.append({
        "pdf": f"{stem}.pdf", "format": res["format"],
        "project_start_pages": CLASSIFIED[stem]["labels"].count("project"), "rows": len(recs),
        "total_matches_plan_years": n_ok, "total_mismatch": n_bad, "plan_years_not_printed": n_blank,
        "total_row_source": "; ".join(f"{k}: {v}" for k, v in how.items()),
    })
SUMMARY = pd.DataFrame(summary)
display(SUMMARY)

def printed_grand_total(stem):
    """Largest 'Grand Total' / 'TOTAL ALL PROJECTS' amount on or just after the summary-table pages."""
    c = CLASSIFIED[stem]
    pages = [i for i, l in enumerate(c["labels"]) if l == "project"]
    best = None
    for i in range(pages[0], min(len(c["texts"]), pages[-1] + 4)):
        flat = " ".join(c["texts"][i].split())
        for m in re.finditer(r"(?:Grand\s+Total|TOTAL\s+ALL\s+PROJECTS(?:\s+All\s+Projects)?)((?:\s+(?:[\d,]{4,}|-))+)", flat, re.I):
            nums = [parse_amount(x) for x in m.group(1).split()]
            nums = [v for v in nums if v]
            if nums and (best is None or nums[-1] > best):
                best = nums[-1]
    return best


b_rows = []
for stem, res in RESULTS.items():
    if res["format"] != "B":
        continue
    parsed = sum(v for r in res["records"] for v in r["_years"].values() if v)
    printed = printed_grand_total(stem)
    b_rows.append({"pdf": f"{stem}.pdf", "rows": len(res["records"]), "sum_of_rows_$K": to_thousands(parsed),
                   "printed_grand_total_$K": to_thousands(printed),
                   "difference_$K": None if printed is None else to_thousands(parsed - printed)})
print("Format B: parsed rows vs printed grand total")
display(pd.DataFrame(b_rows))

null_rates = {}
for stem, res in RESULTS.items():
    df = res["df"]
    if len(df):
        null_rates[stem] = {c: round(100 * (df[c].astype(str).str.strip() == "").mean(), 1) for c in SPEC_COLUMNS}
NULL_RATES = pd.DataFrame(null_rates).T
print("Percent empty per spec column:")
display(NULL_RATES)

# Rows whose plan years don't add up to the printed total (first 40) — inspect these pages in the PDF.
mismatch_rows = []
for stem, res in RESULTS.items():
    for r in res["records"]:
        if r.get("project_total") is None:
            continue
        ys = sorted(r["_years"])
        plan = [r["_years"][y] for y in ys[1:]]
        if plan and None not in plan and abs(sum(plan) - r["project_total"]) > max(2.0, 0.001 * abs(r["project_total"])):
            mismatch_rows.append({"pdf": f"{stem}.pdf", "page": r["source_page"], "project_name": r["project_name"],
                                  "sum_plan_years_$K": to_thousands(sum(plan)),
                                  "printed_total_$K": to_thousands(r["project_total"])})
display(pd.DataFrame(mismatch_rows).head(40))

print("Re-OCR of scanned-sheet fields:", "ON" if REOCR_ACTIVE else "OFF (text layer only)")
for k, v in sorted(REOCR_STATS.items()):
    print(f"  {k}: {v}")
print(f"Repeated project pages dropped: {len(DROPPED_DUPLICATES)}")
display(pd.DataFrame(DROPPED_DUPLICATES))

,pdf,format,project_start_pages,rows,total_matches_plan_years,total_mismatch,plan_years_not_printed,total_row_source
0,2001.pdf,A,198,198,170,1,0,unlabelled total row: 198
1,2002.pdf,A,235,234,224,0,0,unlabelled total row: 234
2,2003.pdf,A,52,52,0,0,48,unlabelled total row: 51; funding table unread...
3,2004.pdf,A,72,72,0,0,71,unlabelled total row: 71; funding table unread...
4,2005.pdf,B,15,103,0,0,0,summary-table TOTAL column: 97; summary-table ...
5,2006.pdf,B,5,48,0,0,0,summary-table TOTAL column: 48
6,2007.pdf,B,7,75,0,0,0,summary-table TOTAL column: 75
7,2008.pdf,B,7,84,0,0,0,summary-table TOTAL column: 84
8,2009.pdf,B,7,90,0,0,0,summary-table TOTAL column: 90
9,2010.pdf,B,6,120,0,0,0,summary-table TOTAL column: 120


Format B: parsed rows vs printed grand total


,pdf,rows,sum_of_rows_$K,printed_grand_total_$K,difference_$K
0,2005.pdf,103,138471.970,138471.970,0.000
1,2006.pdf,48,70954.241,70954.241,0.000
2,2007.pdf,75,98784.697,98784.697,0.000
3,2008.pdf,84,59609.816,59609.816,0.000
4,2009.pdf,90,53733.600,53733.600,0.000
5,2010.pdf,120,52687.502,52687.500,0.002


Percent empty per spec column:


,cip_year,project_type,source_page,department,project_name,project_id,address_location,start_year,end_year,project_description,project_justification,previous_appropriations,project_total
2001,0.0,13.1,0.0,0.0,0.0,16.2,100.0,100.0,100.0,6.1,100.0,49.0,100.0
2002,0.0,7.7,0.0,0.0,0.0,26.1,100.0,100.0,100.0,1.7,100.0,40.2,100.0
2003,0.0,11.5,0.0,0.0,0.0,26.9,100.0,100.0,100.0,0.0,100.0,100.0,100.0
2004,0.0,18.1,0.0,0.0,0.0,23.6,100.0,100.0,100.0,5.6,100.0,100.0,100.0
2005,0.0,100.0,0.0,0.0,0.0,35.0,100.0,100.0,100.0,1.9,100.0,100.0,5.8
2006,0.0,100.0,0.0,0.0,0.0,22.9,100.0,100.0,100.0,0.0,100.0,100.0,0.0
2007,0.0,100.0,0.0,0.0,0.0,41.3,100.0,100.0,100.0,20.0,100.0,100.0,0.0
2008,0.0,100.0,0.0,0.0,0.0,32.1,100.0,100.0,100.0,7.1,100.0,100.0,0.0
2009,0.0,100.0,0.0,0.0,0.0,24.4,100.0,100.0,100.0,4.4,100.0,100.0,0.0
2010,0.0,100.0,0.0,0.0,0.0,45.0,100.0,100.0,100.0,26.7,100.0,100.0,0.0


,pdf,page,project_name,sum_plan_years_$K,printed_total_$K
0,2001.pdf,37,Housing Opportunities for Persons with AIDS,2875.000,2675.000
1,2012.pdf,117,Public Safety Radio Replacement,2400.000,4900.000
2,2012.pdf,118,Swift Water Rescue,0.000,250.000
3,2012.pdf,135,In - Car Camera System,150.000,550.000
4,2012.pdf,137,Police Equipment,1450.000,1950.000
5,2012.pdf,138,Police Station Security Camera Upgrades,0.000,150.000
6,2012.pdf,166,Bike Infrastructure,545.000,645.000
7,2012.pdf,200,Neighborhhod Housing Initiative,7650.000,9300.000
8,2012.pdf,201,Major Development,9510.000,11095.000
9,2012.pdf,202,Property Management and Relocation,2400.000,2800.000


Re-OCR of scanned-sheet fields: ON
  department: re-OCR text used: 557
  description: re-OCR text used: 526
  description: text layer kept: 11
  name: re-OCR rejected (unlike text layer): 1
  name: re-OCR text used: 538
  name: text layer kept: 17
  project_id: filled by re-OCR: 42
  project_total: filled by re-OCR: 75
Repeated project pages dropped: 1


,pdf,dropped_page,kept_page,project_name
0,2002.pdf,154,153,North Side Underpasses - Mayor's Supplement
